# MS-SQL (T-SQL) to Databricks SQL — Reference & Migration Guide

This notebook provides a side-by-side comparison of common MS-SQL / T-SQL syntax and its Databricks SQL equivalent, runnable migration examples, and a deep dive on the `VARIANT` type vs. JSON handling. Use it as a quick reference when porting T-SQL code to Databricks SQL.

## Side-by-Side Quick Reference

### Data Types

| MS-SQL (T-SQL) | Databricks SQL | Notes |
| --- | --- | --- |
| `DATETIME` / `DATETIME2` | `TIMESTAMP` | Also `TIMESTAMP_NTZ` for non-timezone-aware |
| `NVARCHAR(n)` / `VARCHAR(n)` | `STRING` | Databricks `STRING` is unlimited length |
| `BIT` | `BOOLEAN` | |
| `MONEY` / `SMALLMONEY` | `DECIMAL(19,4)` | No native MONEY type |
| `UNIQUEIDENTIFIER` | `STRING` | Store as a UUID-formatted string |
| `XML` | `STRING` | No native XML; use string or VARIANT |
| `JSON` (SQL Server 2022+) | `VARIANT` | Native semi-structured type |
| N/A | `VARIANT`, `ARRAY`, `MAP`, `STRUCT` | Databricks supports complex types natively |

### Common Functions

| MS-SQL (T-SQL) | Databricks SQL | Notes |
| --- | --- | --- |
| `GETDATE()` | `CURRENT_TIMESTAMP()` | |
| `GETUTCDATE()` | `to_utc_timestamp(current_timestamp(), 'UTC')` | `CURRENT_UTC_TIMESTAMP()` available in newer DBR |
| `ISNULL(a, b)` | `COALESCE(a, b)` or `NVL(a, b)` | |
| `LEN(str)` | `LENGTH(str)` or `CHAR_LENGTH(str)` | |
| `CHARINDEX(sub, str)` | `INSTR(str, sub)` | Note: argument order is reversed |
| `LEFT(str, n)` | `LEFT(str, n)` | Same syntax |
| `RIGHT(str, n)` | `RIGHT(str, n)` | Same syntax |
| `CONVERT(type, expr)` | `CAST(expr AS type)` | Also `try_cast()` for safe casts |
| `DATEDIFF(unit, start, end)` | `DATEDIFF(end, start)` (days) or `months_between()` | Different argument order; 2-arg returns days |
| `DATEADD(unit, n, date)` | `DATEADD(date, n)` or `date + INTERVAL n DAYS` | Different signature |
| `FORMAT(value, fmt)` | `DATE_FORMAT(date, fmt)` / `FORMAT_STRING()` | |
| `STRING_AGG(col, sep)` | `ARRAY_JOIN(COLLECT_LIST(col), sep)` | Collect then join |
| `STUFF(str, start, len, repl)` | `OVERLAY(str PLACING repl FROM start [FOR len])` | |
| `NEWID()` | `UUID()` | |
| `TOP n` | `LIMIT n` | `OFFSET ... LIMIT` for pagination; `FETCH FIRST` not available in all DBR |
| `SELECT @@ROWCOUNT` | Not available | Use `COUNT(*)` in a separate query |

### Identifiers & Quoting

| MS-SQL (T-SQL) | Databricks SQL | Notes |
| --- | --- | --- |
| `[column_name]` | `` `column_name` `` | Use backticks instead of brackets |
| `"column_name"` | `"column_name"` | Both support double-quoted identifiers |
| `@variable` | `:param_name` or `{{param}}` widget | No `DECLARE @var` syntax |
| `#TempTable` | `TEMPORARY VIEW` or `TEMP TABLE` | Session-scoped, no `#` prefix |

### Architecture & Control Flow

| MS-SQL (T-SQL) | Databricks SQL | Notes |
| --- | --- | --- |
| Single-node OLTP/OLAP engine | Distributed Spark OLAP engine | |
| `IF...ELSE` / `WHILE` / `BEGIN...END` | Limited procedural support | Stored procedures exist but are Python/SQL-based |
| `TRY...CATCH` | Not available | Use `TRY_*` functions for safe operations |
| Cursors | Not available | Use set-based / window functions |
| `MERGE` | `MERGE INTO` | Delta-specific syntax with schema evolution |
| Stored procedures (T-SQL) | Stored procedures (SQL + Python) | No full T-SQL control flow |
| `IDENTITY(1,1)` | `GENERATE_ALWAYS_AS_IDENTITY()` | |

## Migration Examples

The following cells demonstrate runnable Databricks SQL equivalents of common T-SQL patterns. Each example includes a comment showing the original T-SQL syntax.

### String Functions

Common string operations and their T-SQL to Databricks SQL equivalents.

In [0]:
%sql
-- T-SQL:  SELECT LEN('Hello World')
--          SELECT CHARINDEX('o', 'Hello World')
--          SELECT ISNULL(NULL, 'default')
--          SELECT STUFF('ABCDEF', 3, 2, 'XY')
--          SELECT CONCAT('A', 'B', 'C')

SELECT
  LENGTH('Hello World')                                    AS str_length,        -- replaces LEN()
  INSTR('Hello World', 'o')                                 AS char_position,     -- replaces CHARINDEX (note reversed args)
  COALESCE(NULL, 'default')                                 AS null_replacement,  -- replaces ISNULL()
  OVERLAY('ABCDEF' PLACING 'XY' FROM 3 FOR 2)               AS stuff_equiv,        -- replaces STUFF()
  CONCAT('A', 'B', 'C')                                     AS concatenated,       -- same as T-SQL CONCAT()
  UPPER('hello')                                            AS upper_str,          -- same as T-SQL
  SUBSTRING('Hello World', 1, 5)                             AS substr,             -- same as T-SQL
  TRIM('  spaces  ')                                        AS trimmed,            -- same as T-SQL TRIM()
  REPLACE('Hello World', 'World', 'Databricks')              AS replaced;           -- same as T-SQL

### Date & Time Functions

Date arithmetic and formatting differences between T-SQL and Databricks SQL.

In [0]:
%sql
-- T-SQL:  SELECT GETDATE()
--          SELECT GETUTCDATE()
--          SELECT DATEDIFF(day, '2024-01-01', '2024-03-15')
--          SELECT DATEADD(day, 7, '2024-01-01')
--          SELECT FORMAT(GETDATE(), 'yyyy-MM-dd')
--          SELECT CONVERT(date, GETDATE())

SELECT
  CURRENT_TIMESTAMP()                                              AS current_ts,       -- replaces GETDATE()
  to_utc_timestamp(current_timestamp(), 'UTC')                       AS current_utc,      -- replaces GETUTCDATE()
  CURRENT_DATE()                                                    AS current_date,     -- replaces CONVERT(date, GETDATE())
  DATEDIFF(DATE '2024-03-15', DATE '2024-01-01')                    AS days_diff,        -- note: reversed args vs T-SQL; returns days
  DATE '2024-01-01' + INTERVAL 7 DAYS                                AS add_days,         -- replaces DATEADD(day, 7, ...)
  DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyy-MM-dd')                    AS formatted_date,   -- replaces FORMAT(...)
  DATE '2024-01-01'                                                 AS date_only,        -- replaces CONVERT(date, ...)
  YEAR(CURRENT_DATE())                                              AS current_year,     -- same as T-SQL YEAR()
  MONTH(CURRENT_DATE())                                             AS current_month,    -- same as T-SQL MONTH()
  DAY(CURRENT_DATE())                                               AS current_day;      -- same as T-SQL DAY()

### NULL Handling & Conditional Logic

T-SQL uses `ISNULL()` and `IIF()`; Databricks SQL uses `COALESCE()`, `NVL()`, `NVL2()`, and `IF()`.

In [0]:
%sql
-- T-SQL:  SELECT ISNULL(NULL, 'fallback')
--          SELECT IIF(score >= 60, 'Pass', 'Fail')
--          SELECT CASE WHEN x > 0 THEN 'pos' WHEN x < 0 THEN 'neg' ELSE 'zero' END
--          SELECT COALESCE(NULL, NULL, 'first non-null')

SELECT
  COALESCE(NULL, 'fallback')                               AS coalesce_demo,   -- replaces ISNULL()
  NVL(NULL, 'fallback')                                     AS nvl_demo,         -- Oracle-style null replacement
  NVL2(NULL, 'not-null-value', 'null-value')                AS nvl2_demo,        -- returns value2 if expr1 is NULL
  IF(85 >= 60, 'Pass', 'Fail')                               AS if_demo,          -- replaces IIF()
  CASE
    WHEN 10 > 0 THEN 'positive'
    WHEN 10 < 0 THEN 'negative'
    ELSE 'zero'
  END                                                         AS case_demo,        -- same as T-SQL CASE
  COALESCE(NULL, NULL, 'first non-null')                     AS coalesce_chain,   -- same as T-SQL COALESCE
  -- T-SQL uses TRY_CONVERT for safe casts; Databricks uses try_cast()
  try_cast('not-a-date' AS DATE)                             AS try_cast_demo;    -- returns NULL instead of error

### Top-N Rows & Pagination

T-SQL uses `TOP n` and `OFFSET ... FETCH NEXT ...`; Databricks SQL uses `LIMIT` and `OFFSET ... LIMIT`.

In [0]:
%sql
-- T-SQL:  SELECT TOP 10 * FROM sys.objects
--          SELECT * FROM t ORDER BY id OFFSET 20 ROWS FETCH NEXT 10 ROWS ONLY

-- Databricks SQL: use LIMIT instead of TOP
SELECT * FROM range(100) LIMIT 10;

-- Pagination with OFFSET (Databricks SQL)
-- Equivalent to: SELECT * FROM t ORDER BY id OFFSET 20 ROWS FETCH NEXT 10 ROWS ONLY
SELECT id FROM range(100) ORDER BY id LIMIT 10 OFFSET 20;

-- FETCH FIRST syntax is also documented but not available in all DBR versions
-- SELECT id FROM range(100) ORDER BY id FETCH FIRST 10 ROWS ONLY;

### Identifiers, Temp Tables & MERGE

Key syntax differences for object referencing and data modification.

In [0]:
%sql
-- T-SQL:  SELECT [Order ID], [Customer Name] FROM #TempSales
--          CREATE TABLE #Temp (id INT, name NVARCHAR(50))
--          MERGE target USING source ON target.id = source.id
--            WHEN MATCHED THEN UPDATE SET ...
--            WHEN NOT MATCHED THEN INSERT ...

-- Databricks SQL: backticks instead of brackets, TEMPORARY VIEW instead of #Temp
CREATE OR REPLACE TEMPORARY VIEW temp_sales AS
  SELECT 1 AS `Order ID`, 'Alice' AS `Customer Name`
  UNION ALL
  SELECT 2, 'Bob';

-- Query using backticks for identifiers with spaces
SELECT `Order ID`, `Customer Name` FROM temp_sales;

-- MERGE INTO syntax (requires actual Delta tables, not temp views)
-- T-SQL MERGE is similar but Databricks adds schema evolution support
-- Example (requires real Delta tables in Unity Catalog):
--
-- MERGE INTO catalog.schema.target_table t
-- USING catalog.schema.source_table s
--   ON t.id = s.id
-- WHEN MATCHED THEN
--   UPDATE SET t.name = s.name
-- WHEN NOT MATCHED THEN
--   INSERT (id, name) VALUES (s.id, s.name);

## VARIANT Type vs JSON in Databricks SQL

### What is VARIANT?

`VARIANT` is a native Databricks SQL data type designed for semi-structured data (JSON, Avro, Parquet nested types). Unlike storing JSON as a plain `STRING` and parsing it on every query, `VARIANT` stores the data in an efficient binary representation with a schema-aware layout, enabling faster access and type inference.

### VARIANT vs JSON-as-STRING

| Aspect | JSON as `STRING` | `VARIANT` |
| --- | --- | --- |
| Storage | Raw text | Efficient binary representation |
| Parsing | Must call `from_json()` on every query | Parsed at write time; direct access |
| Path access | `get_json_object(col, '$.key')` | `col:key` (colon notation) |
| Type safety | Returns STRING; must cast manually | Infers types automatically |
| Nested arrays | `from_json()` + schema needed | Direct access with `:path` |
| Schema flexibility | Fixed schema required for parsing | Schemaless; handles any structure |
| Performance | Parse overhead per query | Optimized for repeated reads |

### How T-SQL Handles JSON (for comparison)

T-SQL (SQL Server 2016+) stores JSON as `NVARCHAR` and provides:
- `JSON_VALUE(col, '$.key')` — extract scalar value
- `JSON_QUERY(col, '$.array')` — extract object/array
- `OPENJSON(col)` — shred JSON into rows
- `ISJSON(col)` — validate JSON
- `JSON_MODIFY(col, '$.key', value)` — update a path

Databricks SQL equivalents use `VARIANT` or the `from_json()` / `get_json_object()` family.

### When to use VARIANT vs STRING-based JSON

- Use `VARIANT` when: you store semi-structured data and query it frequently; you need schema flexibility; you deal with nested arrays/maps of unknown shape.
- Use `STRING` + `from_json()` when: you need a fixed, strongly-typed schema; you want explicit control over parsing; the JSON is a small field queried rarely.
- Use `ARRAY<struct>`, `MAP<key, value>`, `STRUCT` when: the schema is well-known and fixed — these give the best performance and type safety.

In [0]:
%sql
-- T-SQL:  SELECT JSON_VALUE(json_col, '$.name'), JSON_QUERY(json_col, '$.tags')
--          SELECT * FROM OPENJSON(json_col) WITH (name NVARCHAR(50) '$.name', age INT '$.age')

-- === Databricks SQL: VARIANT approach ===

-- Create a table with a VARIANT column (parse JSON at write time)
CREATE OR REPLACE TEMPORARY VIEW variant_demo AS
  SELECT parse_json('{"name": "Alice", "age": 30, "tags": ["vip", "early-adopter"], "address": {"city": "Toronto", "zip": "M5V"}}') AS data;

-- Direct path access with colon notation (no need for get_json_object)
SELECT
  data:name                    AS name,           -- extracts a string (typed automatically)
  data:age                     AS age,             -- extracts an integer
  data:tags                    AS tags,            -- extracts an array
  data:tags[0]                 AS first_tag,       -- array element access
  data:address:city            AS city,            -- nested object access
  data:address:zip             AS zip,             -- nested object access
  data:nonexistent             AS missing          -- returns NULL (no error)
FROM variant_demo;

-- Extracting typed values with explicit casting
SELECT
  data:name::STRING             AS name_str,       -- explicit cast to STRING
  data:age::INT                  AS age_int,        -- explicit cast to INT
  try_cast(data:age AS BOOLEAN)  AS age_as_bool     -- try_cast for safe conversion
FROM variant_demo;

In [0]:
%sql
-- === Databricks SQL: STRING-based JSON approach (comparable to T-SQL) ===

-- T-SQL stores JSON as NVARCHAR and uses JSON_VALUE, JSON_QUERY, OPENJSON
-- Databricks SQL uses get_json_object, from_json, and json_array_length

-- Store JSON as a STRING (closest analog to T-SQL's NVARCHAR JSON)
CREATE OR REPLACE TEMPORARY VIEW json_demo AS
  SELECT '{"name": "Bob", "age": 25, "tags": ["new", "trial"], "address": {"city": "San Francisco", "zip": "94101"}}' AS json_str;

In [0]:
%sql
-- Access with get_json_object (similar to JSON_VALUE in T-SQL)
SELECT
  get_json_object(json_str, '$.name')            AS name,        -- like JSON_VALUE()
  get_json_object(json_str, '$.age')             AS age,
  get_json_object(json_str, '$.address.city')    AS city,        -- nested path
  get_json_object(json_str, '$.tags[0]')         AS first_tag,   -- array element
  json_array_length(get_json_object(json_str, '$.tags')) AS tag_count
FROM json_demo;

In [0]:
%sql
-- from_json with explicit schema (similar to OPENJSON ... WITH in T-SQL)
SELECT
  parsed.name      AS name,
  parsed.age       AS age,
  parsed.tags      AS tags,
  parsed.address   AS address
FROM (
  SELECT from_json(json_str, 'STRUCT<name: STRING, age: INT, tags: ARRAY<STRING>, address: STRUCT<city: STRING, zip: STRING>>') AS parsed
  FROM json_demo
);

-- to_json: convert a struct to JSON string (reverse of from_json)
SELECT to_json(struct('Charlie' AS name, 35 AS age)) AS json_output;

### VARIANT vs JSON — Performance & Best Practices

**Performance:**
- `VARIANT` is parsed once at write time and stored in an optimized binary format. Repeated reads of the same field are fast because the parse is already done.
- `STRING` + `from_json()` parses the JSON on every query. For frequently-queried fields, this adds measurable overhead.
- For fixed-schema semi-structured data, `ARRAY<STRUCT<...>>` or `STRUCT<...>` typed columns outperform both `VARIANT` and `STRING`.

**Migration Tip:** If your T-SQL code uses `JSON_VALUE()` or `JSON_QUERY()` extensively, the simplest migration path is:
1. Replace `JSON_VALUE(col, '$.path')` with `col:path` (if using `VARIANT`) or `get_json_object(col, '$.path')` (if still using `STRING`).
2. Replace `OPENJSON(col) WITH (...)` with `from_json(col, 'STRUCT<...>')`.
3. Replace `JSON_MODIFY(col, '$.key', val)` with `to_json(set_field(from_json(col, schema), 'key', val))` or build a new struct.
4. Replace `ISJSON(col)` with `try_parse_json(col) IS NOT NULL` (Databricks SQL).

**Caveat:** `VARIANT` has a max object size of 16 MB (after encoding). `STRING` has no such limit.

## Key Takeaways

1. **Data types map closely but differ in names** — `DATETIME` becomes `TIMESTAMP`, `NVARCHAR` becomes `STRING`, `BIT` becomes `BOOLEAN`.
2. **Function argument order matters** — `DATEDIFF` reverses start/end, `INSTR` reverses substring/string vs T-SQL's `CHARINDEX`.
3. **No `TOP` keyword** — use `LIMIT` or `FETCH FIRST n ROWS ONLY`.
4. **No `DECLARE @var`** — use query parameters (`:param`) or widgets (`{{param}}`).
5. **Backticks instead of brackets** — `` `column` `` replaces `[column]` for identifiers.
6. **`VARIANT` is the modern replacement for JSON-as-string** — use `parse_json()` at write time and `:` path notation for access.
7. **Limited procedural logic** — no `WHILE` loops, cursors, or `TRY...CATCH`. Use set-based operations, `try_cast()` / `try_*` functions, and window functions instead.
8. **Distributed by design** — Databricks SQL is built for large-scale analytics, not OLTP. Queries that run single-node in T-SQL are distributed across a Spark cluster automatically.